# 第3章：预训练实战

## 本章目标
- 在 Shakespeare 数据集上完整训练一个 GPT
- 分析 loss 曲线，理解训练 dynamics
- 实验 learning rate、batch size 对训练的影响
- 理解 Scaling Laws 的基本规律

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch matplotlib
    !wget -q -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## Pretraining = Next-Token Prediction

Pretraining 的目标极其简单：给定前面的 token，预测下一个 token。

这就是语言模型唯一需要做的事。所有的"智能"都涌现自这个简单的目标 + 足够大的数据 + 足够大的模型。

loss function: Cross-Entropy Loss，和分类任务一样。区别在于分类数量 = 词表大小（GPT-2 是 50,257 类）。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPTConfig:
    def __init__(self, vocab_size=50304, block_size=1024,
                 n_layer=12, n_head=12, n_embd=768, dropout=0.1):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.block_size = config.block_size
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.h = nn.ModuleList([Block(config.n_embd, config.n_head,
                                       config.block_size, config.dropout)
                                 for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        tok_emb = self.wte(idx)
        pos_emb = self.wpe(pos)
        x = self.drop(tok_emb + pos_emb)
        for block in self.h:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
with open("input.txt", "r") as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"词表大小: {vocab_size}")
print(f"训练集: {len(train_data):,} tokens, 验证集: {len(val_data):,} tokens")

In [ ]:
@torch.no_grad()
def estimate_loss(model, train_data, val_data, block_size, batch_size, eval_iters=100):
    model.eval()
    losses = {}
    for split, data_source in [("train", train_data), ("val", val_data)]:
        losses_split = torch.zeros(eval_iters)
        for i in range(eval_iters):
            ix = torch.randint(len(data_source) - block_size, (batch_size,))
            x = torch.stack([data_source[j:j+block_size] for j in ix])
            y = torch.stack([data_source[j+1:j+block_size+1] for j in ix])
            _, loss = model(x, y)
            losses_split[i] = loss.item()
        losses[split] = losses_split.mean()
    model.train()
    return losses

def train_gpt(config, train_data, val_data, max_iters=2000, eval_interval=200,
              learning_rate=3e-4, batch_size=64, block_size=256):
    model = GPT(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    train_losses, val_losses, steps = [], [], []

    for iter in range(max_iters):
        if iter % eval_interval == 0:
            losses = estimate_loss(model, train_data, val_data, block_size, batch_size)
            train_losses.append(losses["train"].item())
            val_losses.append(losses["val"].item())
            steps.append(iter)
            print(f"Step {iter:5d} | train loss: {losses['train']:.4f} | val loss: {losses['val']:.4f}")

        ix = torch.randint(len(train_data) - block_size, (batch_size,))
        xb = torch.stack([train_data[i:i+block_size] for i in ix])
        yb = torch.stack([train_data[i+1:i+block_size+1] for i in ix])
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    return model, train_losses, val_losses, steps

In [ ]:
config = GPTConfig(vocab_size=vocab_size, block_size=256,
                   n_layer=6, n_head=6, n_embd=384, dropout=0.2)
model, train_losses, val_losses, steps = train_gpt(
    config, train_data, val_data,
    max_iters=3000, eval_interval=300,
    learning_rate=1e-3, batch_size=64
)

import matplotlib.pyplot as plt
plt.plot(steps, train_losses, label="train")
plt.plot(steps, val_losses, label="val")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.title("Training Loss Curve")
plt.show()

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens=500, temperature=1.0, top_k=None):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

context = torch.zeros((1, 1), dtype=torch.long)
output = generate(model, context, max_new_tokens=500, temperature=0.8, top_k=40)
print("".join([itos[i.item()] for i in output[0]]))

分析生成结果：观察模型学到了什么模式。初期的输出看起来像随机字符，随着训练推进，逐渐出现类似英文单词和句子结构的模式。这就是 next-token prediction 的威力——从简单的预测任务中涌现出语言理解能力。

In [ ]:
import matplotlib.pyplot as plt

results = {}
for lr in [1e-2, 1e-3, 1e-4]:
    cfg = GPTConfig(vocab_size=vocab_size, block_size=128,
                    n_layer=4, n_head=4, n_embd=128, dropout=0.1)
    _, tl, vl, s = train_gpt(cfg, train_data, val_data,
                              max_iters=1000, eval_interval=200,
                              learning_rate=lr, batch_size=32, block_size=128)
    results[lr] = (s, vl)

fig, ax = plt.subplots()
for lr, (s, vl) in results.items():
    ax.plot(s, vl, label=f"lr={lr}")
ax.set_xlabel("Step")
ax.set_ylabel("Val Loss")
ax.legend()
ax.set_title("Learning Rate Comparison")
plt.show()

## Scaling Laws 速览

[Chinchilla 论文](https://arxiv.org/abs/2203.15556) 的核心结论：
1. 给定计算预算 C，最优模型大小 N 和数据量 D 满足 N ∝ C^0.5, D ∝ C^0.5
2. 模型大小和数据量应该等比例增长（很多团队只增大模型，没有相应增加数据）
3. 这意味着：训练一个 1B 参数的模型，大约需要 20B tokens 的数据

实践启示：在数据量有限时，不要盲目增大模型。

## 练习 + 延伸阅读

1. 修改 `max_iters` 和 `learning_rate`，观察 loss 曲线的变化
2. 尝试更大的模型（增加 n_layer 和 n_embd），对比训练速度和效果
3. 用 OpenWebText 子集代替 Shakespeare，观察不同数据分布的影响

- [Chinchilla Scaling Laws](https://arxiv.org/abs/2203.15556)
- [nanoGPT train.py](https://github.com/karpathy/nanoGPT/blob/master/train.py)
- [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361)